# COVID-19 Chest X-Ray Classification
A simple Sequential Dense neural network to classify chest X-rays into **Covid**, **Normal**, or **Viral Pneumonia**.

Dataset: [COVID-19 Image Dataset (Kaggle)](https://www.kaggle.com/datasets/pranavraikokte/covid19-image-dataset)

## 1. Load the data

We load images directly from the `train/` and `test/` folders. Keras automatically uses each subfolder name (`Covid`, `Normal`, `Viral Pneumonia`) as the class label.

In [ ]:
import tensorflow as tf
from tensorflow import keras

train_dir = "/Users/sanjib700/Desktop/My_Projects/Covide_image/Covid19-dataset/train"
test_dir  = "/Users/sanjib700/Desktop/My_Projects/Covide_image/Covid19-dataset/test"

IMG_SIZE = (150, 150)
BATCH_SIZE = 16

train_ds = keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=True,
    seed=42,
)

test_ds = keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False,
)

class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", class_names)

## 2. Preprocess: rescale pixel values

Raw pixel values range from 0–255. We rescale them to 0–1, which helps gradient descent train more smoothly and stably.

In [ ]:
normalization_layer = keras.layers.Rescaling(1.0 / 255)
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
test_ds = test_ds.map(lambda x, y: (normalization_layer(x), y))

## 3. Build the model

A simple **Sequential** model using only `Flatten` + `Dense` layers (same style as the *Hands-On Machine Learning* book's Fashion MNIST example in Chapter 10) — no convolutional layers.

- `Flatten` — unrolls the 150×150×3 image into a single list of 67,500 numbers
- `Dense(300)` and `Dense(100)` — hidden layers, learning patterns from the flattened pixels
- `Dense(num_classes, softmax)` — output layer, one neuron per class, giving a probability for each

In [ ]:
model = keras.Sequential([
    keras.layers.Input(shape=(150, 150, 3)),
    keras.layers.Flatten(),
    keras.layers.Dense(300, activation="relu"),
    keras.layers.Dense(100, activation="relu"),
    keras.layers.Dense(num_classes, activation="softmax"),
])

model.summary()

## 4. Compile the model

- **Loss**: `categorical_crossentropy` — matches our one-hot encoded labels (`label_mode="categorical"`)
- **Optimizer**: `adam` — an adaptive version of gradient descent
- **Metric**: `accuracy` — the main number we'll track

In [ ]:
model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)

## 5. Train the model

We use `EarlyStopping` to automatically stop training if validation accuracy stops improving, and to roll back to the **best** epoch's weights rather than just whichever epoch training happened to end on.

> **Note:** here `test_ds` is being passed as `validation_data` during training. For a more rigorous setup, a separate validation split (carved out of `train_ds`) should be used instead, keeping `test_ds` completely untouched until final evaluation.

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_accuracy",
    patience=5,
    restore_best_weights=True,
)

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=30,
    callbacks=[early_stop],
    verbose=2,
)

## 6. Evaluate on the test set

A single overall accuracy number on the full test set.

In [ ]:
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"Test accuracy: {test_acc:.4f}")

## 7. Quick sanity check: a few sample predictions

Look at the first batch of test images and compare true vs. predicted labels.

In [ ]:
for images, labels in test_ds.take(1):
    preds = model.predict(images, verbose=0)
    for i in range(3):
        true_class = class_names[labels[i].numpy().argmax()]
        pred_class = class_names[preds[i].argmax()]
        confidence = preds[i].max()
        print(f"True: {true_class:16s} Predicted: {pred_class:16s} Confidence: {confidence:.1%}")
    break

## 8. Full test set evaluation: per-class metrics + confusion matrix

A single accuracy number can hide problems (e.g. doing great on one class, poorly on another). This checks **every** test image and breaks results down by class.

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

all_true = []
all_pred = []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    all_true.extend(labels.numpy().argmax(axis=1))
    all_pred.extend(preds.argmax(axis=1))

all_true = np.array(all_true)
all_pred = np.array(all_pred)

print(f"Total test images checked: {len(all_true)}\n")

print("=== Per-class performance ===")
print(classification_report(all_true, all_pred, target_names=class_names))

print("=== Confusion matrix ===")
print("Rows = true class, Columns = predicted class")
cm = confusion_matrix(all_true, all_pred)
print("            ", "  ".join(f"{c[:8]:>8s}" for c in class_names))
for i, row in enumerate(cm):
    print(f"{class_names[i][:12]:12s}", "  ".join(f"{v:8d}" for v in row))

## 9. Predict on new, individual images

A reusable function to load a single image from disk, preprocess it exactly like the training data (resize + rescale), and get the model's prediction.

In [ ]:
def predict_single_image(image_path, model, class_names, img_size=(150, 150)):
    img = keras.utils.load_img(image_path, target_size=img_size)
    img_array = keras.utils.img_to_array(img) / 255.0
    img_array_batch = np.expand_dims(img_array, axis=0)

    prediction = model.predict(img_array_batch, verbose=0)[0]
    predicted_class = class_names[prediction.argmax()]
    confidence = prediction.max()

    print(f"Predicted class: {predicted_class}")
    print(f"Confidence: {confidence:.1%}\n")
    print("All class probabilities:")
    for name, prob in zip(class_names, prediction):
        print(f"  {name:16s}: {prob:.1%}")

    return predicted_class, confidence

# Example usage — replace with a real image path:
# predict_single_image(
#     "/Users/sanjib700/Desktop/My_Projects/Covide_image/Covid19-dataset/test/Covid/some_image.png",
#     model,
#     class_names,
# )

## 10. Predict on every image in a folder

Loop through all images in a given folder (e.g. `test/Covid`), predict each one, and summarize how many were classified correctly — since every image in that specific folder shares the same true label.

In [ ]:
import os

def predict_folder(folder_path, model, class_names, img_size=(150, 150)):
    results = []
    valid_extensions = (".png", ".jpg", ".jpeg")
    image_files = [f for f in os.listdir(folder_path) if f.lower().endswith(valid_extensions)]

    print(f"Found {len(image_files)} images in {folder_path}\n")

    for filename in image_files:
        image_path = os.path.join(folder_path, filename)
        img = keras.utils.load_img(image_path, target_size=img_size)
        img_array = keras.utils.img_to_array(img) / 255.0
        img_array_batch = np.expand_dims(img_array, axis=0)

        prediction = model.predict(img_array_batch, verbose=0)[0]
        predicted_class = class_names[prediction.argmax()]
        confidence = prediction.max()

        results.append({"filename": filename, "predicted": predicted_class, "confidence": confidence})
        print(f"{filename:30s} -> Predicted: {predicted_class:16s} Confidence: {confidence:.1%}")

    return results

# Example usage:
folder_path = "/Users/sanjib700/Desktop/My_Projects/Covide_image/Covid19-dataset/test/Covid"
results = predict_folder(folder_path, model, class_names)

true_label = "Covid"
correct = sum(1 for r in results if r["predicted"] == true_label)
total = len(results)
print(f"\n=== Summary ===")
print(f"Correct: {correct}/{total} ({correct/total:.1%})")